# RESISC45 Knowledge Distillation

Offline KD: SigLIP2 teacher (~93M) -> MobileNetV3-Large student (~4M) on NWPU-RESISC45.

Pipeline: cache teacher logits once (PyTorch) -> train Keras student with KD loss
`L = a*T^2*KL(teacher||student) + (1-a)*CE`. Single deterministic 224px view.

Run cells top-to-bottom. Each section has a verify print.

In [1]:
%pip install torch torchvision --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install tensorflow[and-cuda] tensorboard keras  datasets --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.10.0+cu128 requires nvidia-nccl-cu12==2.27.5; platform_system == "Linux", but you have nvidia-nccl-cu12 2.30.7 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# --- GPU library fix -------------------------------------------------------
# TF 2.21's plain wheel dlopen()s CUDA libs by bare soname and never adds the
# pip nvidia-*-cu12 package dirs to the loader path, so it silently falls back
# to CPU ("Cannot dlopen some GPU libraries"). Preload the bundled CUDA 12 .so
# files (skip the cu13 packages -- TF 2.21 is built against CUDA 12.5 / cuDNN 9)
# so TF finds them. Must run BEFORE `import tensorflow`.
import ctypes, glob, site
for _sp in site.getsitepackages():
    for _so in glob.glob(os.path.join(_sp, "nvidia", "*", "lib", "*.so*")):
        if "/cu13/" in _so:
            continue
        try:
            ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pass
# ---------------------------------------------------------------------------

import tensorflow as tf
tf.config.optimizer.set_jit(False)
print(tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))


I0000 00:00:1782888584.254882   54593 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782888584.287997   54593 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782888584.931367   54593 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2.21.0
Num GPUs Available:  1
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


W0000 00:00:1782888585.878698   54593 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


## 0. Install teacher deps (run once)

In [4]:
%pip install -q torch transformers datasets tqdm

Note: you may need to restart the kernel to use updated packages.


## 1. Config

In [5]:
import os, random, numpy as np

SEED = 42
IMG_SIZE = 224
NUM_CLASSES = 45
BATCH = 64
TEMP = 3.0          # KD temperature
ALPHA = 0.7         # weight on the soft (KD) term
EPOCHS = 50
TEACHER_ID = "prithivMLmods/RESISC45-SigLIP2"
DATASET_ID = "jonathan-roberts1/NWPU-RESISC45"
LOGITS_PATH = "teacher_logits.npy"

random.seed(SEED)
np.random.seed(SEED)

## 2. Data: load + seeded 60/20/20 split

The dataset's `label` feature gives each image an integer in the *dataset's* class
order. We do NOT use that order as canonical -- see section 3.

In [6]:
from datasets import load_dataset

ds = load_dataset(DATASET_ID, split="train")
print(ds)
dataset_names = ds.features["label"].names   # index = dataset label int -> class name
assert len(dataset_names) == NUM_CLASSES, dataset_names
ds_labels = np.array(ds["label"])

N = len(ds)
perm = np.random.default_rng(SEED).permutation(N)
n_test = int(0.2 * N); n_val = int(0.2 * N)
test_idx  = perm[:n_test]
val_idx   = perm[n_test:n_test + n_val]
train_idx = perm[n_test + n_val:]
print(f"N={N}  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

Dataset({
    features: ['image', 'label'],
    num_rows: 31500
})
N=31500  train=18900  val=6300  test=6300


## 3. Teacher + label alignment (the silent killer)

The teacher's 45 logit columns follow `teacher.config.id2label`. We make THAT the
canonical order, and remap the dataset's integer labels into it by matching class
names. If any name fails to match, the assert fires loudly here instead of
silently wrecking accuracy.

In [7]:
import torch
from transformers import AutoImageProcessor, SiglipForImageClassification

processor = AutoImageProcessor.from_pretrained(TEACHER_ID)
teacher = SiglipForImageClassification.from_pretrained(TEACHER_ID)
device = "cuda" if torch.cuda.is_available() else "cpu"
teacher.to(device).eval()

id2label = teacher.config.id2label
teacher_names = [id2label[i] for i in range(len(id2label))]   # canonical order

def norm(s):
    return s.lower().replace("_", " ").replace("-", " ").strip()

teacher_norm = {norm(n): i for i, n in enumerate(teacher_names)}
ds2teacher = np.array([teacher_norm[norm(n)] for n in dataset_names], dtype=np.int64)
assert len(set(ds2teacher.tolist())) == NUM_CLASSES, "label alignment failed!"

labels = ds2teacher[ds_labels]   # canonical (teacher-order) label per image
print("label alignment OK")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

label alignment OK


## 4. Cache teacher logits (deterministic, single view)

Runs the teacher once over every image with its own HF processor (resize 224,
normalize 0.5). Cached to disk so a kernel restart won't recompute.

In [8]:
if os.path.exists(LOGITS_PATH):
    teacher_logits = np.load(LOGITS_PATH)
    print("loaded cached logits", teacher_logits.shape)
else:
    from tqdm.auto import tqdm
    teacher_logits = np.zeros((N, NUM_CLASSES), dtype=np.float32)
    with torch.no_grad():
        for start in tqdm(range(0, N, BATCH)):
            imgs = [im.convert("RGB") for im in ds[start:start + BATCH]["image"]]
            inputs = processor(images=imgs, return_tensors="pt").to(device)
            out = teacher(**inputs).logits
            teacher_logits[start:start + len(imgs)] = out.float().cpu().numpy()
    np.save(LOGITS_PATH, teacher_logits)
    print("cached", teacher_logits.shape)

loaded cached logits (31500, 45)


**Verify alignment:** teacher top-1 on the test split should be ~0.95.

In [9]:
t_pred = teacher_logits.argmax(1)
print(f"teacher test top-1: {(t_pred[test_idx] == labels[test_idx]).mean():.4f}")

teacher test top-1: 0.9444


## 5. tf.data pipelines: (image, teacher_logits) -> label

MobileNetV3 (`include_preprocessing=True`) does its own rescaling, so we feed raw
0-255 pixels resized to 224. Generator indexes the HF dataset by id, keeping
images, logits and labels aligned.

In [10]:
import tensorflow as tf
import keras
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(indices, training):
    idx = indices.astype(np.int64)
    def gen():
        for i in idx:
            img = np.asarray(ds[int(i)]["image"].convert("RGB"), dtype=np.uint8)
            yield img, teacher_logits[i], labels[i]
    sig = (tf.TensorSpec((None, None, 3), tf.uint8),
           tf.TensorSpec((NUM_CLASSES,), tf.float32),
           tf.TensorSpec((), tf.int64))
    d = tf.data.Dataset.from_generator(gen, output_signature=sig)
    def prep(img, tl, y):
        img = tf.image.resize(tf.cast(img, tf.float32), (IMG_SIZE, IMG_SIZE))
        return (img, tl), y
    d = d.map(prep, num_parallel_calls=AUTOTUNE)
    if training:
        d = d.shuffle(2048, seed=SEED)
    return d.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_idx, True)
val_ds   = make_ds(val_idx, False)
test_ds  = make_ds(test_idx, False)

W0000 00:00:1782888594.249122   54593 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1782888594.251751   54593 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1782888594.252479   54593 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 94492 MB memory:  -> device: 0, name: NVIDIA RTX PRO 6000 Blackwell Server Edition, pci bus id: 0000:11:00.0, compute capability: 12.0a


## 6. Student: MobileNetV3-Large + 45-class logit head

In [11]:
base = keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False, weights="imagenet", include_preprocessing=True)
x = keras.layers.GlobalAveragePooling2D()(base.output)
x = keras.layers.Dropout(0.2)(x)
out = keras.layers.Dense(NUM_CLASSES)(x)            # logits, no activation
student = keras.Model(base.input, out, name="student_mnv3l")
print(f"student params: {student.count_params():,}")

student params: 965,085


## 7. Distiller (KD loss in train_step)

In [12]:
class Distiller(keras.Model):
    def __init__(self, student, T=3.0, alpha=0.7):
        super().__init__()
        self.student = student
        self.T = T
        self.alpha = alpha
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.soft_tracker = keras.metrics.Mean(name="soft_kd")   # KD term (T^2 * KL)
        self.hard_tracker = keras.metrics.Mean(name="hard_ce")   # ground-truth CE term
        self.acc = keras.metrics.SparseCategoricalAccuracy(name="acc")

    @property
    def metrics(self):
        return [self.loss_tracker, self.soft_tracker, self.hard_tracker, self.acc]

    def _loss(self, y, t_logits, s_logits):
        ce = tf.reduce_mean(
            keras.losses.sparse_categorical_crossentropy(y, s_logits, from_logits=True))
        t_soft = tf.nn.softmax(t_logits / self.T)
        s_logsoft = tf.nn.log_softmax(s_logits / self.T)
        kl = tf.reduce_mean(
            tf.reduce_sum(t_soft * (tf.math.log(t_soft + 1e-8) - s_logsoft), axis=1))
        soft = (self.T ** 2) * kl
        total = self.alpha * soft + (1 - self.alpha) * ce
        return total, soft, ce

    def _update(self, y, s_logits, total, soft, ce):
        self.loss_tracker.update_state(total)
        self.soft_tracker.update_state(soft)
        self.hard_tracker.update_state(ce)
        self.acc.update_state(y, s_logits)
        return {m.name: m.result() for m in self.metrics}

    def train_step(self, data):
        (img, t_logits), y = data
        with tf.GradientTape() as tape:
            s_logits = self.student(img, training=True)
            total, soft, ce = self._loss(y, t_logits, s_logits)
        grads = tape.gradient(total, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))
        return self._update(y, s_logits, total, soft, ce)

    def test_step(self, data):
        (img, t_logits), y = data
        s_logits = self.student(img, training=False)
        total, soft, ce = self._loss(y, t_logits, s_logits)
        return self._update(y, s_logits, total, soft, ce)

## 8. Train (AdamW + cosine LR with warmup)

Logs to TensorBoard under `logs/kd/<timestamp>`: total loss, `soft_kd`, `hard_ce`
and `acc` for both train and validation. Launch with `tensorboard --logdir logs/kd`.

In [13]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=2,
    verbose=1,
    min_lr=1e-6
)

In [14]:
import datetime

steps = (len(train_idx) + BATCH - 1) // BATCH
lr = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5, warmup_target=1e-3, warmup_steps=steps,
    decay_steps=steps * EPOCHS, alpha=0.01)
distiller = Distiller(student, T=TEMP, alpha=ALPHA)
distiller.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4))

logdir = os.path.join("logs", "mobileV3S", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard = keras.callbacks.TensorBoard(log_dir=logdir)
print("logging to", logdir)
hist = distiller.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                     callbacks=[tensorboard, early_stopping])

logging to logs/mobileV3S/20260701-064955
Epoch 1/50


/opt/conda/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1782888605.306022   56015 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1782888606.693921   55544 service.cc:153] XLA service 0x7f433003c480 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782888606.693950   55544 service.cc:161]   StreamExecutor [0]: NVIDIA RTX PRO 6000 Blackwell Server Edition, Compute Capability 12.0a (Driver: 13.1.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1782888607.003041   55544 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:17828886

    294/Unknown 53s 36ms/step - acc: 0.5416 - hard_ce: 1.7754 - loss: 7.1831 - soft_kd: 9.5006

I0000 00:00:1782888650.505113   55545 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_43258__.569
I0000 00:00:1782888659.995254   55545 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1782888660.175868   58781 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_6', 60 bytes spill stores, 60 bytes spill loads

I0000 00:00:1782888660.222320   58774 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_6', 40 bytes spill stores, 40 bytes spill loads



    296/Unknown 80s 129ms/step - acc: 0.5430 - hard_ce: 1.7674 - loss: 7.1495 - soft_kd: 9.4562

...

296/296 ━━━━━━━━━━━━━━━━━━━━ 15s 45ms/step - acc: 0.9750 - hard_ce: 0.0756 - loss: 0.1669 - soft_kd: 0.2060 - val_acc: 0.9511 - val_hard_ce: 0.1501 - val_loss: 0.3288 - val_soft_kd: 0.4053
Epoch 48/50
294/296 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - acc: 0.9761 - hard_ce: 0.0728 - loss: 0.1676 - soft_kd: 0.2083

I0000 00:00:1782889412.825134   56013 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 3897503698033457181
I0000 00:00:1782889412.825178   56013 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 1053698232334251077
I0000 00:00:1782889412.825183   56013 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 14339832817952911329


296/296 ━━━━━━━━━━━━━━━━━━━━ 15s 48ms/step - acc: 0.9761 - hard_ce: 0.0731 - loss: 0.1678 - soft_kd: 0.2084 - val_acc: 0.9513 - val_hard_ce: 0.1501 - val_loss: 0.3288 - val_soft_kd: 0.4054
Epoch 49/50
296/296 ━━━━━━━━━━━━━━━━━━━━ 15s 45ms/step - acc: 0.9757 - hard_ce: 0.0747 - loss: 0.1662 - soft_kd: 0.2054 - val_acc: 0.9508 - val_hard_ce: 0.1501 - val_loss: 0.3283 - val_soft_kd: 0.4046
Epoch 50/50
292/296 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - acc: 0.9761 - hard_ce: 0.0732 - loss: 0.1644 - soft_kd: 0.2034

I0000 00:00:1782889442.457253   56013 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 3897503698033457181
I0000 00:00:1782889442.457294   56013 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 1053698232334251077
I0000 00:00:1782889442.457298   56013 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 14339832817952911329


296/296 ━━━━━━━━━━━━━━━━━━━━ 15s 47ms/step - acc: 0.9761 - hard_ce: 0.0730 - loss: 0.1646 - soft_kd: 0.2038 - val_acc: 0.9503 - val_hard_ce: 0.1506 - val_loss: 0.3287 - val_soft_kd: 0.4050


In [15]:
distiller.summary()

Model: "distiller"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ student_mnv3l (Functional)      │ (None, 45)             │       965,085 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,871,032 (10.95 MB)

 Trainable params: 952,973 (3.64 MB)

 Non-trainable params: 12,112 (47.31 KB)

 Optimizer params: 1,905,947 (7.27 MB)

In [16]:
%reload_ext tensorboard
%tensorboard --logdir logs/kd

## 9. Evaluate: student top-1 + fidelity to teacher

Fidelity = how often the student agrees with the teacher (top-1) and the mean
KL between their softmax outputs on the held-out test set.

In [17]:
s_logits_all, y_all = [], []
for (img, _tl), y in test_ds:
    s_logits_all.append(student(img, training=False).numpy())
    y_all.append(y.numpy())
s_logits_all = np.concatenate(s_logits_all)
y_all = np.concatenate(y_all)

s_pred = s_logits_all.argmax(1)
t_logits_test = teacher_logits[test_idx]
t_pred = t_logits_test.argmax(1)

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

ps, pt = softmax(s_logits_all), softmax(t_logits_test)
kl = (pt * (np.log(pt + 1e-8) - np.log(ps + 1e-8))).sum(1).mean()

print(f"student test top-1     : {(s_pred == y_all).mean():.4f}")
print(f"teacher test top-1     : {(t_pred == y_all).mean():.4f}")
print(f"top-1 agreement        : {(s_pred == t_pred).mean():.4f}")
print(f"mean KL(teacher||stud) : {kl:.4f}")

student test top-1     : 0.9422
teacher test top-1     : 0.9444
top-1 agreement        : 0.9543
mean KL(teacher||stud) : 0.0902


## 10. Save the trained student

In [18]:
os.makedirs("models", exist_ok=True)
student.save("models/student_mnv3l.keras")
print("saved models/student_mnv3l.keras")

saved models/student_mnv3l.keras
